In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch

cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

In [7]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [9]:
device = 'cuda'

In [3]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
# X_test = scaler.transform(X_test)

X_train = torch.FloatTensor(X_train)
X_valid = torch.FloatTensor(X_valid)

y_train = torch.FloatTensor(y_train).reshape(-1, 1)
y_valid = torch.FloatTensor(y_valid).reshape(-1, 1)

In [6]:
train_data = TensorDataset(X_train, y_train)
valid_data = TensorDataset(X_valid, y_valid)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32, shuffle=False)

In [22]:
class CancerClassifier(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features, 16),
            nn.ReLU(),
            nn.Dropout(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Dropout(),
        )
        self.output_layer = nn.Linear(8, 1)
    
    def forward(self, X):
        deep = self.deep_stack(X)
        return self.output_layer(deep)

In [23]:
def train(model, optimizer, criterion, train_loader, n_epochs):
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        mean_train_loss = total_loss / len(train_loader)

        model.eval()
        total_valid_loss = 0
        total_correct = 0      # New: Keep track of right answers
        total_samples = 0      # New: Keep track of total patients tested
        
        with torch.no_grad():
            for X_batch, y_batch in valid_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                
                # 1. Get raw predictions and calculate loss
                raw_pred = model(X_batch)
                loss = criterion(raw_pred, y_batch)
                total_valid_loss += loss.item()
                
                # 2. ACCURACY MATH: Convert raw numbers to probabilities (0 to 1)
                probabilities = torch.sigmoid(raw_pred)
                
                # 3. Round to 0 or 1 (If > 0.5, guess 1. Else guess 0)
                guesses = probabilities.round()
                
                # 4. Count how many guesses match the actual y_batch
                correct_guesses = (guesses == y_batch).sum().item()
                
                total_correct += correct_guesses
                total_samples += len(y_batch) # Add the number of patients in this batch
                
        mean_valid_loss = total_valid_loss / len(valid_loader)
        
        # Calculate final accuracy percentage for the epoch
        accuracy = (total_correct / total_samples) * 100
        
        # Print the report!
        print(f"Epoch {epoch + 1:02d}/{n_epochs} | Train Loss: {mean_train_loss:.4f} | Valid Loss: {mean_valid_loss:.4f} | Accuracy: {accuracy:.2f}%")

In [24]:
model = CancerClassifier(n_features=30).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()

In [25]:
train(model, optimizer, criterion, train_loader, n_epochs =100)

Epoch 01/100 | Train Loss: 0.7154 | Valid Loss: 0.6877 | Accuracy: 38.60%
Epoch 02/100 | Train Loss: 0.6850 | Valid Loss: 0.6435 | Accuracy: 61.40%
Epoch 03/100 | Train Loss: 0.6402 | Valid Loss: 0.5899 | Accuracy: 85.96%
Epoch 04/100 | Train Loss: 0.5765 | Valid Loss: 0.5189 | Accuracy: 93.86%
Epoch 05/100 | Train Loss: 0.5358 | Valid Loss: 0.4419 | Accuracy: 95.61%
Epoch 06/100 | Train Loss: 0.4580 | Valid Loss: 0.3663 | Accuracy: 95.61%
Epoch 07/100 | Train Loss: 0.4100 | Valid Loss: 0.3030 | Accuracy: 96.49%
Epoch 08/100 | Train Loss: 0.3723 | Valid Loss: 0.2529 | Accuracy: 96.49%
Epoch 09/100 | Train Loss: 0.3524 | Valid Loss: 0.2141 | Accuracy: 96.49%
Epoch 10/100 | Train Loss: 0.2931 | Valid Loss: 0.1847 | Accuracy: 96.49%
Epoch 11/100 | Train Loss: 0.2983 | Valid Loss: 0.1624 | Accuracy: 96.49%
Epoch 12/100 | Train Loss: 0.2462 | Valid Loss: 0.1429 | Accuracy: 97.37%
Epoch 13/100 | Train Loss: 0.2468 | Valid Loss: 0.1279 | Accuracy: 97.37%
Epoch 14/100 | Train Loss: 0.2531 | Va

In [27]:
def predict(model, new_data, device):
    # 1. Put the model in test mode (turns off Dropout!)
    model.eval()
    
    # 2. Turn off the Autograd engine (saves memory)
    with torch.no_grad():
        # 3. Teleport the new data to the GPU
        new_data = new_data.to(device)
        
        # 4. Get the raw numbers from the model
        raw_pred = model(new_data)
        
        # 5. Convert the raw numbers into probabilities (0.0 to 1.0)
        probabilities = torch.sigmoid(raw_pred)
        
        # 6. Round the probabilities to exactly 0 or 1
        final_guesses = probabilities.round()
        
        # 7. Teleport the answers back to the CPU and convert to a standard NumPy array
        return final_guesses.cpu().numpy()

In [ ]:
# Grab the first 5 rows of your validation data
sample_patients = X_valid[:5]
actual_answers = y_valid[:5].numpy()

# Pass them into your new predict function
ai_predictions = predict(model, sample_patients, device)

print("AI's Diagnoses:")
print(ai_predictions.flatten())

print("\nActual Diagnoses (The Truth):")
print(actual_answers.flatten())

AI's Diagnoses:
[1. 0. 0. 1. 1. 0. 0. 0. 1. 1. 1. 0. 1. 0. 1. 0. 1. 1. 1. 0. 1. 1. 0. 1.
 1. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 0. 1. 0. 1. 1. 0. 1. 1. 1. 1. 1. 1.
 1. 1. 0. 0. 1. 1. 1.]

Actual Diagnoses (The Truth):
[1. 0. 0. 1. 1. 0. 0. 0. 1. 1. 1. 0. 1. 0. 1. 0. 1. 1. 1. 0. 0. 1. 0. 1.
 1. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 0. 1. 0. 1. 1. 0. 1. 1. 1. 1. 1. 1.
 1. 1. 0. 0. 1. 1. 1.]
